# 初始化模型
LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象(1.0版本之前的方式)
## init_chat_model
官方最推荐的方式就是使用init_chat_model函数

### 基于名称推断模型提供商

使用init_chat_model函数，你需要从LangChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。

LangChain支持的模型列表参考官网链接：
https://docs.langchain.com/oss/python/integrations/providers/overview

接下来你要做的事情包括：
- 安装模型依赖：uv add langchain langchain-deepseek
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [1]:
from dotenv import load_dotenv
load_dotenv()
# 导入langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型
# 参数model用来指定模型名称，LangChain会根据模型名字自动设定base_url,并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

In [2]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


### 自定义模型提供商
init_chat_model默认会根据模型名称自动确定模型的提供者的base_url,并从env读取api_key,但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其他模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max,它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key

In [3]:
# 我们收到加载环境变量中的base_url和api_key
import os
from langchain.chat_models import init_chat_model
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-max", # 模型名称，这里可以自定义，我们用的阿里云的qwen-max
    model_provider="openai", # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai,所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [4]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：

- temperature：控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越稳定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...



## 使用model类

其实init_chat_model函数底层就是帮我们利用Model类创建对象，但只支持有限的模型。
而在langchain社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。
具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat

例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1.首先，我们需要安装依赖LangChain社区依赖(不再继续维护了)：

```bash
    uv add langchain-community
```
   阿里云百炼依赖：
```bash
    uv add dashscope
```
2.然后，我们就可以使用Model类初始化模型了

In [5]:
from langchain_community.chat_models.tongyi import ChatTongyi
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("DASHSCOPE_API_KEY")
# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max",
    api_key=api_key #这种方式必须指定api_key了，国庆期间还不需要指定的，这个大家去了解一下就好了
    # 其他模型参数...
)

C:\Users\15893\AppData\Local\Temp\ipykernel_62884\1427248441.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_models.tongyi import ChatTongyi


In [6]:
# 打印结果
print(type(model))

<class 'langchain_community.chat_models.tongyi.ChatTongyi'>


# 访问模型

LangChain提供了两个不同的函数来访问模型：

- invoke:阻塞式访问
- stream:流式访问

## 方式一：invoke

invoke函数是阻塞式调用，需要等待模型生成全部结果才返回，等待时间较长。

In [7]:
# 通过invoke函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁？")

In [8]:
# 查看响应内容
print(response)

content='我是Qwen，由阿里云开发的预训练语言模型。我的设计目的是帮助用户生成各种类型的文本，如文章、故事、诗歌、故事等，并提供信息咨询、解答问题等服务。有什么我可以帮你的吗？' additional_kwargs={} response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': '5d52d9b7-db68-9b09-a781-4ceb2b6d1c9b', 'token_usage': {'input_tokens': 11, 'output_tokens': 49, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 60}} id='lc_run--01a1a9b7-fe45-7072-990b-e8b846b217d6-0' tool_calls=[] invalid_tool_calls=[]


In [9]:
# 调用invoke函数，传入消息列表
response = model.invoke([
    {"role": "system", "content": "你扮演王者荣耀的英雄角色，以英雄的口吻回答用户的问题。"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)

我是王者荣耀中的英雄，你想要了解哪一位英雄呢？比如我可以说，我是手持巨剑，以德服人的亚瑟；或是脚踏机关，灵动飘逸的公孙离。请告诉我你更想了解谁。


## 方式二:stream
invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个词
避坑：如果是通过Model类返回的model去调用这个stream,其底层默认不走流式。必须在初始化model的时候在里面显示写stream=True,包括也需要在里面指定api_key，国庆期间还不需要指定api_key的，哎

In [10]:
# 通过.stream函数实现流式访问
stream = model.stream("你是谁？")

In [11]:
# 打印stream类型（生成器类型，生成式一个token一个token拼接返回）
print(type(stream))

<class 'generator'>


In [12]:
for chunk in stream:
    print(chunk.content, end=" | ",flush=True) #是否能够看到每个token之间的|还得看你这个model是怎么来的

我是Qwen，阿里云开发的一款超大规模语言模型。我的目标是成为一款能够广泛服务于社会的AI助手，帮助人们提高工作效率和生活质量。无论是信息查询、语言理解、文本生成还是其他复杂任务，我都会尽力提供支持。有什么我可以帮到你的吗？ | 

# 在智能体中访问模型
本节我们学习如何在智能体中使用模型。
## 创建智能体

Langchain提供了一个create_agent函数用来快速创建智能体。调用create_agent需要指定一个模型。有两种选择：

- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型

In [13]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [14]:
# 2.指定model名称，由Langchain自动初始化模型
agent = create_agent(model="deepseek-chat")

## 调用智能体

智能体调用与模型调用类似，也支持两种方式：

- invoke:阻塞式调用
- stream:流式访问

但需要注意的是，智能体调用时需要传入一个dict,其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [15]:
response = agent.invoke({
    "messages" : [{"role": "user", "content": "你是谁？ "}]
})

print(response)

{'messages': [HumanMessage(content='你是谁？ ', additional_kwargs={}, response_metadata={}, id='e2c1d75e-70fd-4ce6-b9bc-1918b26e86fc'), AIMessage(content='你好！我是 DeepSeek，由深度求索公司创造的 AI 助手！😊\n\n我可以帮你解答各种问题、进行对话、处理文本任务，还能接收你上传的图片并分析其中的可见信息。我的一些特点包括：\n\n✨ **完全免费**使用\n📱 支持 Web 和 App 端（可在官方应用商店下载）\n📄 支持上传图片、PDF、Word、Excel、PPT 等文件并读取其中的文字信息\n🔗 可以阅读链接内容\n🌐 支持联网搜索（需要手动开启）\n🎤 App 端支持语音输入\n💪 拥有 1M 超长上下文，可以一次性处理像《三体》三部曲体量的书籍\n\n我的知识截止日期是 2026 年 2 月，会尽我所能为你提供准确、有用的帮助！\n\n有什么我可以帮你的吗？随时问我！🌟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 187, 'prompt_tokens': 7, 'total_tokens': 194, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 7}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'd70bae2a-eb77-4dfa

### 流式访问

In [16]:
# 通过stream函数实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？ "}]},
    stream_mode="messages"
)
print(type(messages))

<class 'generator'>


In [17]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:
        print(token.content, end="|", flush=True)

你好|！|我是| Deep|Se|ek|，|由|深度|求|索|公司|创造的| AI| |助手|。|😊|

|我可以|帮你|解答|问题|、|处理|文字|信息|、|分析|文档|等|。|我|支持|接收|图片|、|PDF|、|Word|、|Excel| |等|文件|，|并|从中|读取|文字|信息|，|也能|识别|图片|中的|可见|内容|。|同时|，|我|支持|联网|搜索|（|需要|你|手动|开启|），|上下文|长度|达到| |1|M|，|可以|一次性|处理|像|《|三|体|》|三部|曲|那样|体|量的|长|文本|。

|而且|我现在|是|**|完全|免费|**|的|！|无论是|网页|版|还是| App| |都可以|直接|使用|。|App| |端|还|支持|语音|输入|，|非常|方便|。

|有什么|我可以|帮|你的|吗|？|尽管|问我|吧|！|✨|